# Módulo 3 - IA Generativa

### 1. Introdução
**Contexto**

As autoridades querem relatórios claros e visuais para comunicar com decisores e cidadãos. Pretende-se usar IA Generativa para resumir resultados dos Módulos 1 e 2 e propor recomendações.

**Objetivos de Aprendizagem**
1.	Explorar engenharia de prompts para síntese textual;
2.	Criar conteúdos inovadores, mas responsáveis;
3.	Discutir limitações, riscos e implicações socioeconómicas.

**Tarefas**
1.	Desenvolver um script ou notebook gen_report.py que recebe dados em JSON/CSV e gera:
    1.	resumo executivo (≤ 200 palavras);
    2.	2-3 recomendações de ação;
    3.	secção “limitações e riscos”.
2.	Testar ≥ 3 variantes de prompts e comentar diferenças.
3.	Incluir avaliação crítica: transparência, enviesamentos, possíveis “alucinações”.



In [ ]:
import json
import os
import sys
from pathlib import Path
from IPython.display import Markdown, display

import anthropic
import pandas as pd

# # Adiciona o diretório do módulo ao path para importar gen_report
# MODULE3_DIR = Path(".").resolve()  # ajustar se necessário
# if str(MODULE3_DIR) not in sys.path:
#     sys.path.insert(0, str(MODULE3_DIR))

# Importar funções do gen_report.py
from gen_report import load_facts, generate, PROMPTS, ReportFacts

print("✓ Imports concluídos")
print(f"  Variantes disponíveis: {list(PROMPTS.keys())}")

✓ Imports concluídos
  Variantes disponíveis: ['baseline', 'estruturado', 'critico_etico']


## 3. Configuração da API Key

A chave deve estar na variável de ambiente `ANTHROPIC_API_KEY`.
Se não estiver definida, o script corre em **modo offline** (template determinístico).

In [ ]:
api_key = os.environ.get("ANTHROPIC_API_KEY", "")

if api_key:
    print("✓ ANTHROPIC_API_KEY encontrada — modo API ativo")
    client = anthropic.Anthropic(api_key=api_key)
else:
    print("⚠ ANTHROPIC_API_KEY não definida — modo offline (template estático)")
    print("  Para ativar: export ANTHROPIC_API_KEY='sk-ant-...' (Linux/Mac)")
    print("              ou setx ANTHROPIC_API_KEY 'sk-ant-...' (Windows)")

## 4. Carregamento e Agregação de Factos

A função `load_facts()` lê os três ficheiros de output dos módulos anteriores e constrói
um objeto `ReportFacts` com os números que o LLM vai usar.

In [ ]:
# Caminhos — ajustar à estrutura do projeto
ALERTS_CSV  = Path("../Module_1/resultados_alertas.csv")
RULES_JSON  = Path("../Module_1/regras.json")
METRICS_CSV = Path("../Module_2/resultados/metrics.csv")

facts = load_facts(ALERTS_CSV, RULES_JSON, METRICS_CSV)

# Visualizar o bloco de factos que o LLM vai receber
facts_dict = facts.to_dict()
print(json.dumps(facts_dict, ensure_ascii=False, indent=2))

In [ ]:
# Resumo legível dos factos
n        = facts.n_observations
levels   = facts.alerts_by_level
n_alert  = sum(v for k, v in levels.items() if k != "NORMAL")

print("=" * 50)
print("FACTOS CARREGADOS")
print("=" * 50)
print(f"Cidades    : {', '.join(facts.cities)}")
print(f"Período    : {facts.period}")
print(f"Observações: {n}")
print(f"Com alerta : {n_alert} ({100*n_alert/n:.1f} %)")
print()
print("Distribuição de risco:")
for k, v in facts.alerts_by_level.items():
    print(f"  {k:<10}  {v:>5}  ({100*v/n:.1f} %)")
print()
print("Top regras disparadas:")
for r in facts.top_rules:
    print(f"  {r['id']:<30}  {r['count']} ocorrências")
print()
print("% alertas por cidade:")
for city, pct in facts.pct_alerts_by_city.items():
    print(f"  {city:<15}  {pct:.1f} %")
print()
print("Métricas de classificação (melhor modelo primeiro):")
for r in sorted(facts.classification_metrics, key=lambda x: x["f1"], reverse=True):
    flag = " ← melhor F1" if r["modelo"] == facts.best_classifier else ""
    print(f"  {r['modelo']:<25}  F1={r['f1']:.3f}  AUC={r['roc_auc']:.3f}  recall={r['recall']:.3f}{flag}")
print()
print("Métricas de regressão (NO₂):")
for r in sorted(facts.regression_metrics, key=lambda x: x["r2"], reverse=True):
    flag = " ← melhor R²" if r["modelo"] == facts.best_regressor else ""
    print(f"  {r['modelo']:<25}  R²={r['r2']:.3f}  MAE={r['mae']:.2f} µg/m³{flag}")

## 5. As Três Variantes de Prompt

Todas as variantes recebem **o mesmo bloco de factos JSON** — o que varia são as
instruções dadas ao modelo. O objetivo é perceber como a engenharia do prompt
afeta estrutura, clareza, fidelidade e adequação ao público.

| Variante | Persona | Público-alvo | Ênfase |
|---|---|---|---|
| `baseline` | Assistente genérico | Qualquer | Formato livre |
| `estruturado` | Analista técnico | Proteção Civil | Precisão + fontes regulamentares |
| `critico_etico` | Analista de políticas públicas | Decisores + cidadãos | Clareza + ética + grupos de risco |

In [ ]:
# Mostrar as instruções de cada prompt (sem o bloco de factos, que é longo)
for nome, template in PROMPTS.items():
    instrucoes = template.split("{facts}")[0].strip()
    print(f"{'='*60}")
    print(f"  VARIANTE: {nome.upper()}")
    print(f"{'='*60}")
    print(instrucoes)
    print()

## 6. Geração dos Relatórios

Para cada variante, construímos o prompt completo e enviamos à API.
O resultado é guardado em dicionário para comparação posterior.

In [ ]:
outputs = {}  # { variante: texto_gerado }

for variante in PROMPTS:
    print(f"A gerar variante '{variante}'...", end=" ", flush=True)
    texto = generate(facts, variante)
    outputs[variante] = texto
    n_words = len(texto.split())
    print(f"✓  ({n_words} palavras)")

print("\nGeração concluída para todas as variantes.")

---
### 6.1 Variante Baseline

In [ ]:
display(Markdown(outputs["baseline"]))

---
### 6.2 Variante Estruturado

In [ ]:
display(Markdown(outputs["estruturado"]))

---
### 6.3 Variante Crítico-Ético

In [ ]:
display(Markdown(outputs["critico_etico"]))

## 7. Análise Comparativa das Variantes

Para comparar objetivamente os três outputs, avaliamos automaticamente
indicadores simples: presença de cabeçalhos Markdown, menção a fontes
regulamentares, citação de métricas e tamanho do resumo executivo.

In [ ]:
import re

def analisar_output(texto: str) -> dict:
    """Indicadores heurísticos de qualidade do output."""
    linhas = texto.splitlines()

    # Cabeçalhos Markdown (## ou ###)
    cabecalhos = [l.strip() for l in linhas if l.strip().startswith("##")]

    # Fontes regulamentares mencionadas
    fontes = []
    for src in ["Diretiva 2008", "OMS 2021", "IPMA", "DGS", "2008/50/CE"]:
        if src.lower() in texto.lower():
            fontes.append(src)

    # Métricas numéricas mencionadas
    metricas = []
    for m in ["F1", "R²", "recall", "ROC-AUC", "accuracy", "MAE", "MSE"]:
        if m.lower() in texto.lower():
            metricas.append(m)

    # Grupos de risco
    grupos = []
    for g in ["asmáticos", "idosos", "crianças", "vulneráveis", "trabalhadores"]:
        if g.lower() in texto.lower():
            grupos.append(g)

    # Palavras do resumo executivo (até ao próximo ##)
    resumo_match = re.search(r"## Resumo executivo(.*?)(?=\n##|\Z)", texto, re.DOTALL | re.IGNORECASE)
    n_palavras_resumo = len(resumo_match.group(1).split()) if resumo_match else None

    # Cumpre limite 200 palavras?
    cumpre_limite = (n_palavras_resumo is not None and n_palavras_resumo <= 200)

    return {
        "cabeçalhos": cabecalhos,
        "fontes_regulamentares": fontes,
        "métricas_citadas": metricas,
        "grupos_de_risco": grupos,
        "palavras_resumo": n_palavras_resumo,
        "cumpre_200_palavras": cumpre_limite,
        "total_palavras": len(texto.split()),
    }

# Analisar cada output
analises = {v: analisar_output(t) for v, t in outputs.items()}

# Tabela comparativa
print(f"{'Critério':<35} {'baseline':>12} {'estruturado':>13} {'critico_etico':>14}")
print("-" * 78)

criterios = [
    ("Nº de cabeçalhos Markdown",     lambda a: len(a["cabeçalhos"])),
    ("Fontes regulamentares",          lambda a: len(a["fontes_regulamentares"])),
    ("Métricas ML citadas",            lambda a: len(a["métricas_citadas"])),
    ("Grupos de risco mencionados",    lambda a: len(a["grupos_de_risco"])),
    ("Palavras no resumo executivo",   lambda a: a["palavras_resumo"] or "—"),
    ("Cumpre ≤ 200 palavras",          lambda a: "✓" if a["cumpre_200_palavras"] else "✗"),
    ("Total de palavras no relatório", lambda a: a["total_palavras"]),
]

for nome_c, fn in criterios:
    vals = [fn(analises[v]) for v in ["baseline", "estruturado", "critico_etico"]]
    print(f"{nome_c:<35} {str(vals[0]):>12} {str(vals[1]):>13} {str(vals[2]):>14}")

In [ ]:
# Detalhe: fontes e grupos de risco encontrados
for variante, a in analises.items():
    print(f"\n── {variante.upper()} ──")
    print(f"  Cabeçalhos  : {a['cabeçalhos']}")
    print(f"  Fontes      : {a['fontes_regulamentares']}")
    print(f"  Métricas    : {a['métricas_citadas']}")
    print(f"  Grupos risco: {a['grupos_de_risco']}")

## 8. Guardar Relatórios em Ficheiros Markdown

In [ ]:
OUT_DIR = Path(".")  # diretório de output (mesmo que o notebook)

for variante, texto in outputs.items():
    path = OUT_DIR / f"report_{variante}.md"
    path.write_text(texto, encoding="utf-8")
    print(f"  Guardado: {path}  ({len(texto.split())} palavras)")

print("\n✓ Todos os relatórios guardados.")

## 9. Avaliação Crítica

### 9.1 Comparação Qualitativa das Variantes

| Critério | Baseline | Estruturado | Crítico-ético |
|---|---|---|---|
| Cumpre estrutura pedida (resumo + rec. + limitações) | Parcial | ✓ | ✓ |
| Cita fontes regulamentares (Diretiva 2008/50/CE, IPMA) | ✗ | ✓ | ✓ |
| Identifica grupo de risco específico | ✗ | Parcial | ✓ |
| Adequado a público não-técnico | Médio | Baixo | Alto |
| Reflexão ética explícita (viés, desigualdade) | ✗ | ✗ | ✓ |
| Risco de alucinação (escala estimada) | Médio | Baixo | Baixo |
| Densidade de informação útil | Baixa | Alta | Alta |
| Adequação a decisor municipal | Baixa | Média | **Alta** |

**Conclusão:** A variante `critico_etico` é a mais completa para o contexto de Proteção Civil:
inclui gatilho quantitativo, fonte regulamentar *e* grupo de risco em cada recomendação,
e discute explicitamente a incerteza e o impacto desigual dos alertas.

---

### 9.2 Transparência e Enviesamentos

**O que o sistema faz bem:**
- Toda a informação factual (números, métricas, regras) vem dos Módulos 1 e 2 — o LLM
  não inventa valores.
- A abordagem de *grounding* (factos em JSON) torna o sistema auditável: qualquer número
  no relatório pode ser rastreado até à fonte original.

**Enviesamentos identificados:**
1. **Sobre-interpretação qualitativa.** Quando o prompt não especifica critérios,
   o LLM tende a classificar métricas como "aceitáveis" ou "robustas" sem justificação.
   Exemplo: R² = 0.72 descrito como "bom" — que pode ser insuficiente em contexto clínico.
2. **Viés de confirmação.** O modelo tende a construir narrativas coerentes mesmo quando
   os dados são escassos, o que pode parecer mais confiante do que é justificado.
3. **Impacto desigual não modelado.** Os alertas assumem implicitamente que toda a
   população tem igual capacidade de resposta — não é verdade (acesso a carro, teletrabalho, etc.).

---

### 9.3 Risco de Alucinação e Mitigações

Mesmo com *grounding*, observámos três padrões de risco:

**1. Inferência sobre causa não sustentada pelos dados.**
O modelo pode sugerir que Lisboa tem mais alertas por "tráfego mais denso" —
hipótese plausível mas não provada pelos dados.
*Mitigação:* a variante `critico_etico` instrui o modelo a usar "pode" e a indicar
"não disponível nos dados" quando não há suporte factual.

**2. Generalização de grupos de risco.**
Quando o prompt pede "grupo de risco", o modelo lista grupos canónicos da literatura
(asmáticos, idosos) sem dados específicos sobre esses grupos na amostra.
*Mitigação:* indicar explicitamente no prompt que os grupos provêm de guidelines
clínicas (OMS, DGS), não dos dados observados.

**3. Extrapolação sazonal.**
O dataset cobre setembro-outubro. O modelo pode generalizar recomendações
para outras estações sem aviso explícito.
*Mitigação:* o bloco de factos inclui o período exato; os prompts estruturado
e crítico-ético instruem a mencionar a cobertura limitada nas limitações.

---

### 9.4 Implicações Socioeconómicas

- **Acesso desigual:** alertas de qualidade do ar têm impacto assimétrico —
  quem tem recursos pode adaptar-se (teletrabalho, purificadores, transporte privado);
  quem não tem, não pode.
- **Responsabilidade humana:** nenhum alerta deve ser comunicado ao público sem
  revisão por equipa técnica da Proteção Civil. O sistema é apoio à decisão,
  não decisor autónomo.
- **Literacia digital:** a comunicação deve chegar a populações com menor literacia.
  O relatório `critico_etico` sugere canais múltiplos e linguagem simples.

---

### 9.5 Limitações do Sistema

| Limitação | Impacto | Mitigação possível |
|---|---|---|
| Dataset limitado (2 meses, 2 cidades) | Modelos não generalizáveis para inverno/verão | Alargar cobertura temporal |
| Desbalanço de classes (~10 % 'má') | Accuracy enganadora; KNN recall=0.45 | Usar F1 e recall como métrica principal |
| R² = 0.72 para NO₂ | 28 % da variância não explicada | Adicionar features (volume tráfego, direção vento) |
| LLM não tem memória entre chamadas | Não aprende com correções humanas | RAG ou fine-tuning futuro |
| Custo de API por chamada | Inviável para relatórios em tempo real | Cache de factos + geração batch |

## 10. Conclusão

O Módulo 3 demonstrou como a **engenharia de prompts** muda radicalmente a qualidade
e adequação de um relatório gerado por LLM, mesmo usando os mesmos factos numéricos:

- A variante **baseline** mostra o piso: texto correto mas sem estrutura nem fontes.
- A variante **estruturado** é a melhor para audiências técnicas que precisam de
  precisão e citação de métricas.
- A variante **crítico-ético** é a recomendada para contextos de políticas públicas:
  é a única que nomeia grupos de risco específicos, discute desigualdade de acesso
  e trata os modelos explicitamente como ferramentas de apoio.

A arquitetura de *grounding* (LLM vê apenas factos pré-calculados, não o dataset)
é a principal mitigação de alucinação — mas não a elimina completamente. 
**A revisão humana é parte obrigatória do pipeline.**